# Access and inspect SASBDB data

This notebook retrieves an entry summary and downloadable files from the official [SASBDB](https://www.sasbdb.org/) REST API and entry page. It uses `SASDYU2` as the initial public test case. Change `ACCESSION` to another seven-character SASBDB code when needed.

The downloaded experimental curve is plotted as $I(q)$ versus $q$. SASBDB reports the angular unit in the entry metadata; do not assume the unit when comparing with GROMACS-SWAXS.

In [ ]:
from __future__ import annotations

import json
import re
import shutil
import urllib.request
import zipfile
from html.parser import HTMLParser
from pathlib import Path
from urllib.parse import urljoin, urlparse

import matplotlib.pyplot as plt
import pandas as pd

BASE_URL = "https://www.sasbdb.org"
ACCESSION = "SASDYU2"
DATA_ROOT = Path("data/sasbdb")
DOWNLOAD_ZIP_BUNDLE = True

ACCESSION = ACCESSION.strip().upper()
if not re.fullmatch(r"SAS[A-Z0-9]{4}", ACCESSION):
    raise ValueError("Expected a seven-character SASBDB code such as SASDYU2.")

ENTRY_DIR = DATA_ROOT / ACCESSION
ENTRY_DIR.mkdir(parents=True, exist_ok=True)
ENTRY_DIR

In [ ]:
class LinkCollector(HTMLParser):
    def __init__(self):
        super().__init__()
        self.hrefs = []

    def handle_starttag(self, tag, attrs):
        if tag.lower() == "a":
            href = dict(attrs).get("href")
            if href:
                self.hrefs.append(href)


def request_bytes(url: str, timeout: int = 60) -> bytes:
    request = urllib.request.Request(
        url, headers={"User-Agent": "sasbdb-access-notebook/0.1"}
    )
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return response.read()


def request_json(url: str) -> dict:
    return json.loads(request_bytes(url).decode("utf-8"))


def discover_downloads(code: str) -> list[str]:
    entry_url = f"{BASE_URL}/data/{code}/"
    parser = LinkCollector()
    parser.feed(request_bytes(entry_url).decode("utf-8", errors="replace"))
    allowed_suffixes = {".dat", ".ift", ".fit", ".pdb", ".mrc", ".sascif", ".fasta", ".log", ".zip"}
    urls = []
    for href in parser.hrefs:
        url = urljoin(BASE_URL, href)
        suffix = Path(urlparse(url).path).suffix.lower()
        if suffix in allowed_suffixes and code.lower() in url.lower():
            urls.append(url)
    return sorted(set(urls))


def download_file(url: str, destination: Path, overwrite: bool = False) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and not overwrite:
        return destination
    temporary = destination.with_suffix(destination.suffix + ".part")
    with urllib.request.urlopen(
        urllib.request.Request(url, headers={"User-Agent": "sasbdb-access-notebook/0.1"}),
        timeout=120,
    ) as response, temporary.open("wb") as output:
        shutil.copyfileobj(response, output)
    temporary.replace(destination)
    return destination

## Retrieve metadata and identify available files

In [ ]:
summary_url = f"{BASE_URL}/rest-api/entry/summary/{ACCESSION}/"
metadata = request_json(summary_url)

metadata_path = ENTRY_DIR / f"{ACCESSION}_summary.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

download_urls = discover_downloads(ACCESSION)
pd.DataFrame({"filename": [Path(urlparse(url).path).name for url in download_urls], "url": download_urls})

In [ ]:
project = metadata.get("project", {})
experiment = metadata.get("experiment", {})
instrument = experiment.get("instrument", {})
sample = experiment.get("sample", {})
molecules = sample.get("molecule", [])

overview = {
    "accession": metadata.get("code"),
    "title": project.get("title"),
    "status": metadata.get("status"),
    "curve_type": metadata.get("type_of_curve"),
    "angular_unit": metadata.get("angular_unit"),
    "intensity_unit": metadata.get("intensity_unit"),
    "beamline": instrument.get("beamline_name"),
    "facility": instrument.get("name"),
    "molecules": [m.get("long_name") for m in molecules],
    "concentration_max": experiment.get("concentration_max"),
    "guinier_Rg": metadata.get("guinier_rg"),
    "P_r_Rg": metadata.get("pddf_rg"),
    "Dmax": metadata.get("pddf_dmax"),
}
pd.Series(overview, name="value").to_frame()

## Download the entry

By default, this downloads the SASBDB ZIP bundle. Set `DOWNLOAD_ZIP_BUNDLE = False` to download the discovered files individually. Existing files are retained unless `overwrite=True` is passed to `download_file`.

In [ ]:
downloaded = []
if DOWNLOAD_ZIP_BUNDLE:
    zip_url = f"{BASE_URL}/media/zip_directories/{ACCESSION}.zip"
    zip_path = download_file(zip_url, ENTRY_DIR / f"{ACCESSION}.zip")
    with zipfile.ZipFile(zip_path) as archive:
        # Reject unsafe archive paths before extraction.
        target_root = ENTRY_DIR.resolve()
        for member in archive.infolist():
            target = (ENTRY_DIR / member.filename).resolve()
            if target_root not in target.parents and target != target_root:
                raise ValueError(f"Unsafe ZIP member: {member.filename}")
        archive.extractall(ENTRY_DIR)
    downloaded.append(zip_path)
else:
    for url in download_urls:
        filename = Path(urlparse(url).path).name
        downloaded.append(download_file(url, ENTRY_DIR / filename))

print(f"Saved {len(downloaded)} download(s) under {ENTRY_DIR.resolve()}")
sorted(str(path.relative_to(ENTRY_DIR)) for path in ENTRY_DIR.rglob("*") if path.is_file())

## Load and plot the experimental SAXS curve

In [ ]:
data_candidates = list(ENTRY_DIR.rglob(f"{ACCESSION}.dat"))
if not data_candidates:
    raise FileNotFoundError(f"No {ACCESSION}.dat file was found under {ENTRY_DIR}")

curve = pd.read_csv(
    data_candidates[0],
    sep=r"\s+",
    comment="#",
    names=["q", "I", "sigma_I"],
    usecols=[0, 1, 2],
)
curve = curve.apply(pd.to_numeric, errors="coerce").dropna()
curve.head()

In [ ]:
q_unit = metadata.get("angular_unit", "reported SASBDB unit")
intensity_unit = metadata.get("intensity_unit", "reported SASBDB unit")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.errorbar(curve["q"], curve["I"], yerr=curve["sigma_I"], fmt=".", ms=2, lw=0.5, alpha=0.8)
ax.set_yscale("log")
ax.set_xlabel(f"q [{q_unit}]")
ax.set_ylabel(f"I(q) [{intensity_unit}]")
ax.set_title(f"SASBDB {ACCESSION}: experimental scattering curve")
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

## Unit-preparation helper for GROMACS-SWAXS

GROMACS typically uses inverse nanometers for $q$, while SASBDB entries often use inverse angstroms. Because $1\ \mathrm{\AA}^{-1}=10\ \mathrm{nm}^{-1}$, the following column is convenient for later curve comparison. Intensity scaling is deliberately not applied here because SASBDB may report arbitrary or absolute intensity units.

In [ ]:
curve_for_swaxs = curve.copy()
if str(q_unit).replace(" ", "").lower() in {"1/a", "a^-1", "1/å", "å^-1"}:
    curve_for_swaxs["q_nm^-1"] = 10.0 * curve_for_swaxs["q"]
else:
    curve_for_swaxs["q_nm^-1"] = curve_for_swaxs["q"]
    print(f"Check q conversion manually; SASBDB reports angular_unit={q_unit!r}.")
curve_for_swaxs.head()

### Next planned step

The next iteration can add entry search/filtering, automatic metadata normalization, Guinier and $P(r)$ analysis, and comparison with FoXS or GROMACS-SWAXS curves.